In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

mutation_matrix_path = Path(
    "../data/processed/gbm_primary_profiled_mutation_matrix.tsv"
)

mutation_matrix = pd.read_csv(
    mutation_matrix_path,
    sep="\t",
    index_col=0
)

print("Mutation matrix shape:",
      mutation_matrix.shape)

print("Unique values:",
      np.unique(mutation_matrix.to_numpy()))

Mutation matrix shape: (390, 13909)
Unique values: [0 1]


In [2]:
clinical_path = Path(
    "../data/processed/gbm_primary_clinical.tsv"
)

primary_clinical = pd.read_csv(
    clinical_path,
    sep="\t"
)

primary_sample_ids = set(
    primary_clinical["Sample ID"]
)

sequenced_path = Path(
    "../data/raw/gbm_tcga_pan_can_atlas_2018_cases_sequenced.txt"
)

lines = sequenced_path.read_text().splitlines()

case_ids_line = next(
    line for line in lines
    if line.startswith("case_list_ids:")
)

sequenced_sample_ids = set(
    case_ids_line
    .split(":", 1)[1]
    .strip()
    .split("\t")
)

expression_path = Path(
    "../data/raw/gbm_tcga_pan_can_atlas_2018_mrna_rsem.txt"
)

expression = pd.read_csv(
    expression_path,
    sep="\t",
    low_memory=False
)

expression_sample_ids = set(
    expression.columns[2:]
)

multiomics_primary_ids = sorted(
    primary_sample_ids
    & sequenced_sample_ids
    & expression_sample_ids
)

print(
    "Multi-omics primary patients:",
    len(multiomics_primary_ids)
)

Multi-omics primary patients: 152


In [3]:
mutation_multiomics = mutation_matrix.loc[
    multiomics_primary_ids
].copy()

print(
    "Mutation matrix for multi-omics cohort:",
    mutation_multiomics.shape
)

print(
    "Missing requested patients:",
    len(
        set(multiomics_primary_ids)
        - set(mutation_matrix.index)
    )
)

Mutation matrix for multi-omics cohort: (152, 13909)
Missing requested patients: 0


In [4]:
mutation_multiomics_t = (
    mutation_multiomics.T
)

print(
    "Transposed mutation matrix:",
    mutation_multiomics_t.shape
)

print(
    "Patients:",
    mutation_multiomics_t.shape[1]
)

print(
    "Genes:",
    mutation_multiomics_t.shape[0]
)

Transposed mutation matrix: (13909, 152)
Patients: 152
Genes: 13909


In [5]:
# Re-create expression_status from expression data for downstream integration
expression_multiomics = expression[
    ["Hugo_Symbol", "Entrez_Gene_Id"] + multiomics_primary_ids
]

expression_with_symbols = expression_multiomics[
    expression_multiomics["Hugo_Symbol"].notna()
    & (expression_multiomics["Hugo_Symbol"] != "")
]

expression_gene_matrix = (
    expression_with_symbols
    .groupby("Hugo_Symbol")[multiomics_primary_ids]
    .mean()
)

expression_log2 = np.log2(expression_gene_matrix + 1)
unique_values_per_gene = expression_log2.nunique(axis=1)
constant_genes = unique_values_per_gene[unique_values_per_gene == 1].index
expression_log2_filtered = expression_log2.drop(index=constant_genes)

gene_median = expression_log2_filtered.median(axis=1)
gene_mad = (
    expression_log2_filtered
    .sub(gene_median, axis=0)
    .abs()
    .median(axis=1)
)

nonzero_mad_genes = gene_mad[gene_mad > 0].index
expression_for_scoring = expression_log2_filtered.loc[nonzero_mad_genes]
gene_median_nonzero = gene_median.loc[nonzero_mad_genes]
gene_mad_nonzero = gene_mad.loc[nonzero_mad_genes]

expression_robust_z = (
    expression_for_scoring
    .sub(gene_median_nonzero, axis=0)
    .div(gene_mad_nonzero, axis=0)
    * 0.6745
)

expression_delta_log2 = (
    expression_for_scoring
    .sub(gene_median_nonzero, axis=0)
)

final_high_expression_mask = (
    (expression_robust_z >= 3.5)
    & (expression_delta_log2 >= 1.0)
)

final_low_expression_mask = (
    (expression_robust_z <= -3.5)
    & (expression_delta_log2 <= -1.0)
)

expression_status = pd.DataFrame(
    0,
    index=expression_robust_z.index,
    columns=expression_robust_z.columns,
    dtype="int8"
)

expression_status[final_high_expression_mask] = 1
expression_status[final_low_expression_mask] = -1

print("Expression status shape:", expression_status.shape)

Expression status shape: (17639, 152)


In [6]:
mutation_genes = set(
    mutation_multiomics_t.index
)

expression_genes = set(
    expression_status.index
)

shared_genes = (
    mutation_genes
    & expression_genes
)

mutation_only_genes = (
    mutation_genes
    - expression_genes
)

expression_only_genes = (
    expression_genes
    - mutation_genes
)

print(
    "Mutation genes:",
    len(mutation_genes)
)

print(
    "Expression-scored genes:",
    len(expression_genes)
)

print(
    "Shared genes:",
    len(shared_genes)
)

print(
    "Mutation-only genes:",
    len(mutation_only_genes)
)

print(
    "Expression-only genes:",
    len(expression_only_genes)
)

Mutation genes: 13909
Expression-scored genes: 17639
Shared genes: 11568
Mutation-only genes: 2341
Expression-only genes: 6071
